# Stacking Ensemble Model

Stacking is an ensemble learning technique that combines multiple base models
to improve prediction performance.

In this project:
- Base Models: Random Forest, XGBoost
- Meta Model: Logistic Regression

The base models make predictions, and the meta model learns from those predictions
to make the final decision.

In [1]:
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, roc_auc_score,
                             RocCurveDisplay)

In [6]:
X_train = joblib.load("../models/X_train.pkl")
X_test  = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test  = joblib.load("../models/y_test.pkl")
rf      = joblib.load("../models/random_forest_model.pkl")
xgb     = joblib.load("../models/xgboost_model.pkl")
print("Loaded. Train:", X_train.shape)

Loaded. Train: (565424, 51)


In [3]:
stack_model = StackingClassifier(
    estimators=[
        ('rf',  rf),
        ('xgb', xgb)
    ],
    final_estimator=LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight='balanced'
    ),
    cv=10,
    passthrough=True,
    n_jobs=-1
)

stack_model.fit(X_train, y_train)
joblib.dump(stack_model, "../models/stacking_model.pkl")
print("Stacking model saved.")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Stacking model saved.


In [7]:
y_proba = stack_model.predict_proba(X_test)[:, 1]

print(f"{'Thresh':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 45)
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.20, 1.00, 0.05):
    yp = (y_proba >= t).astype(int)
    pr = precision_score(y_test, yp, zero_division=0)
    re = recall_score(y_test, yp, zero_division=0)
    f1 = f1_score(y_test, yp, zero_division=0)
    print(f"{t:>8.2f} {accuracy_score(y_test,yp):>8.3f} {pr:>8.3f} {re:>8.3f} {f1:>8.3f}")
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"\nPeak F1: {best_f1:.3f} at threshold {best_thresh:.2f}")
print(f"XGB F1=0.547 — Stacking above XGB: {best_f1 > 0.547}")

  Thresh      Acc     Prec      Rec       F1
---------------------------------------------
    0.20    0.738    0.050    0.971    0.095
    0.25    0.765    0.055    0.965    0.105
    0.30    0.786    0.060    0.959    0.113
    0.35    0.801    0.064    0.955    0.120
    0.40    0.815    0.068    0.948    0.127
    0.45    0.859    0.085    0.914    0.155
    0.50    0.906    0.118    0.861    0.207
    0.55    0.938    0.164    0.820    0.273
    0.60    0.957    0.218    0.788    0.342
    0.65    0.968    0.274    0.767    0.404
    0.70    0.975    0.329    0.748    0.457
    0.75    0.978    0.364    0.728    0.486
    0.80    0.981    0.401    0.714    0.514
    0.85    0.983    0.444    0.697    0.543
    0.90    0.985    0.492    0.678    0.570
    0.95    0.988    0.559    0.638    0.596

Peak F1: 0.596 at threshold 0.95
XGB F1=0.547 — Stacking above XGB: True


In [8]:
THRESHOLD = best_thresh
y_pred    = (y_proba >= THRESHOLD).astype(int)

print(f"===== Stacking Model Results (threshold={THRESHOLD:.2f}) =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

joblib.dump(THRESHOLD, "../models/stack_threshold.pkl")
print(f"Threshold saved.")

===== Stacking Model Results (threshold=0.95) =====
Accuracy : 0.988
Precision: 0.559
Recall   : 0.638
F1 Score : 0.596
ROC-AUC  : 0.963
Confusion Matrix:
[[138333   1013]
 [   727   1284]]
Threshold saved.
